# Analyse du Data Drift

Le but est de comparer les données d’entraînement avec les données de production en utilisant Evidently.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from evidently import Report
from evidently.presets import DataDriftPreset

## Chargement des données

Les données de référence viennent du fichier CSV et les données de production sont stockées dans PostgreSQL.

In [2]:
if os.path.exists("/.dockerenv"):
    load_dotenv("docker.env")
else:
    load_dotenv("local.env")

df_train = pd.read_csv("df_total_clean.csv")
engine = create_engine(os.getenv("URLBDD"))
df_production = pd.read_sql('SELECT * FROM "predictions_inputs"', con=engine)

df_train = df_train.drop(columns=["SK_ID_CURR"])
df_production = df_production.drop(columns=["SK_ID_CURR"])

## Simulation du drift

Je garde uniquement les 25 % de clients avec les revenus les plus élevés en production pour simuler une différence avec les données d’entraînement.

In [3]:
seuil = df_production["AMT_INCOME_TOTAL"].quantile(0.75)
clients_riches = df_production[df_production["AMT_INCOME_TOTAL"] > seuil]
df_train_sample = df_train.sample(n=20000, random_state=42)

print("Clients sélectionnés :", len(clients_riches))

Clients sélectionnés : 19


## Rapport Evidently

Evidently compare les distributions des variables entre les deux jeux de données.

In [4]:
report = Report(metrics=[DataDriftPreset()])
resultat = report.run(reference_data=df_train_sample, current_data=clients_riches)
resultat.save_html("drift_report.html")

c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning:

invalid value encountered in divide

c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning:

invalid value encountered in divide



## Résultats et conclusion

Dans le rapport obtenu :

- **117 variables** ont été analysées.
- **65 variables** présentent un drift, soit **55,56 %**.
- Evidently détecte donc un **Data Drift global**.

Le drift est volontairement provoqué en sélectionnant les clients les plus riches. Le résultat montre que le système détecte une différence de distribution.

Le rapport détaillé est disponible dans `drift_report.html`.